In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import pandas as pd

from astropy.io import fits 

import matplotlib.pyplot as plt

import mekuria_imara_pipeline as mip

In [6]:
HI_CUBE_120p = "../data/m33_C+D+tp_himidres_120pc.fits" 
# HI_CUBE_30p = "/Volumes/imara-lab/M33/HI/M33_A+B+C+D_HI_0p4kms_feathered_K_reimaged.fits"
# CATALOG   = "corbelli_tab5.txt"
# D_M33     = 859.0 * u.kpc          # distance used in the notebook (859000 pc)
# FGND_CHAN = 60                     # first 60 channels = MW foreground (notebook [60:])
# N_HI_WNM  = 0.57                   # cm^-3, warm-neutral-medium density (paper)
# MU        = 1.36                   # mean molecular weight incl. He (paper mass defn)
# C_MOI     = 0.4                    # moment-of-inertia coeff for uniform sphere (paper)
# NHI_COEFF = 1.82e18               # cm^-2 / (K km/s), optically-thin 21-cm

In [7]:
fits.open(HI_CUBE)

In [ ]:
fits.open(HI_CUBE_30p)

In [ ]:
def run(nmax=None, snr=5.0, out="hi_table2_m33.csv"):
    # -- load catalog (Corbelli+2017 Table 5) --
    cat = pd.read_csv(CATALOG, delim_whitespace=True)
    if nmax:
        cat = cat.iloc[:nmax]

    # -- open cube (memmap; trim MW foreground) --
    hdu = fits.open(HI_CUBE, memmap=True)[0]
    hdr = hdu.header
    wcs2d = WCS(hdr).celestial
    vax = ((np.arange(hdr["NAXIS3"]) - hdr["CRPIX3"]) * hdr["CDELT3"]
          + hdr["CRVAL3"]) / 1e3                          # km/s
    vax = vax[FGND_CHAN:]
    pc_per_pix = (np.deg2rad(abs(hdr["CDELT2"])) * D_M33).to(u.pc).value
    ny, nx = hdr["NAXIS2"], hdr["NAXIS1"]

    rows = []
    for _, gmc in cat.iterrows():
        name = f"GMC-{int(gmc['N'])}"
        R_pc = R_HI_cylinder(gmc["MH2"]).value                     # [1]
        r_pix = R_pc / pc_per_pix

        # [2] aperture: circular mask around the GMC centre
        xc, yc = wcs2d.wcs_world2pix(gmc["RAdeg"], gmc["DEdeg"], 0)
        xc, yc = float(xc), float(yc)
        x0, x1 = max(0, int(xc - r_pix - 1)), min(nx, int(xc + r_pix + 2))
        y0, y1 = max(0, int(yc - r_pix - 1)), min(ny, int(yc + r_pix + 2))
        if x1 - x0 < 3 or y1 - y0 < 3:
            continue

        sub = np.asarray(hdu.data[FGND_CHAN:, y0:y1, x0:x1], dtype=float)
        yy, xx = np.mgrid[y0:y1, x0:x1]
        dx_pc = (xx - xc) * pc_per_pix
        dy_pc = (yy - yc) * pc_per_pix
        ap = (dx_pc**2 + dy_pc**2) <= R_pc**2                      # cylinder footprint

        # [3] kinematic association from aperture-summed spectrum
        spec = np.nansum(sub * ap[None, :, :], axis=(1, 2))
        vbar, sig = associate_velocity(spec, vax, snr_min=snr)
        if vbar is None:
            continue
        vmask = np.abs(vax - vbar) <= 2 * sig                      # v_bar +/- 2 sigma
        if vmask.sum() < 2:
            continue

        subw = sub[vmask]
        vchan = vax[vmask]

        # [4] HI mass over the associated window (optically thin)
        m0 = moment0(subw, vchan)                                  # K km/s
        N_HI = NHI_COEFF * m0                                      # cm^-2
        pix_area = (pc_per_pix * u.pc).to(u.cm).value ** 2
        M_HI = (MU * const.m_p.to(u.Msun).value
                * np.nansum(N_HI[ap] * pix_area))                  # Msun

        # [5] velocity field
        vfield = first_moment(subw, vchan)

        # [6] plane fit -> Omega, theta
        fit = fit_plane(vfield, dx_pc, dy_pc, ap)
        if fit is None:
            continue
        Wx, Wy, theta, px, py, pv = fit
        Omega = np.hypot(Wx, Wy)                                   # km/s/pc

        # [7] planarity
        rho = planarity_pearson(Wx, Wy, px, py, pv)

        # [8] specific angular momentum
        j = specific_angular_momentum(R_pc, Omega)

        rows.append({
            "Name": name,
            "M_HI_1e4": M_HI / 1e4,
            "R_HI_pc": R_pc,
            "Omega_HI_1e-2": Omega * 1e2,
            "theta_HI_deg": theta,
            "rho_HI": rho,
            "j_HI_1e24": j / 1e24,
        })

    tab = pd.DataFrame(rows)
    tab.to_csv(out, index=False)
    return tab


def _fmt(tab):
    disp = tab.copy()
    disp.columns = ["Name", "M_HI", "R_HI", "Omega_HI", "theta_HI", "rho_HI", "j_HI"]
    return disp.to_string(
        index=False,
        float_format=lambda v: f"{v:8.3g}",
    )


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--nmax", type=int, default=None,
                    help="only process the first N GMCs (testing)")
    ap.add_argument("--snr", type=float, default=5.0,
                    help="min peak/noise for a GMC to be kept")
    ap.add_argument("--out", default="hi_table2_m33.csv")
    args = ap.parse_args()

    table = run(nmax=args.nmax, snr=args.snr, out=args.out)
    print(f"\nTable 2 (HI-only) -- M33 GMCs  [n={len(table)}]")
    print("  M_HI [1e4 Msun]  R_HI [pc]  Omega_HI [1e-2 km/s/pc]  "
          "theta [deg]  rho  j_HI [1e24 cm^2/s]\n")
    print(_fmt(table.head(25)))
    print(f"\nsaved -> {args.out}")
